## HDB Resale Flat Prices - ETL Pipeline
Notebook runs the whole pipeline end to end from grabbing the raw files to creating the tables the Data Science team needs.

Architecture (Medallion Architecture)

| **Layer** | **Output group** | **What it contains**                                                                        |
|-----------|------------------|---------------------------------------------------------------------------------------------|
| Bronze    | Raw              | Files queried and downloaded from data.gov.sg (source) exactly as received. Never modified. |
| Silver    | Cleaned          | Combined and later validated rows with remaining lease and resale-price anomaly flags.      |
| Silver    | Quarantined      | Rows that failed the rejection rules, each with the appropriate rejection reason.           |
| Gold      | Transformed      | Cleaned rows with a unique identifier based on the requirements.                            |
| Gold      | Hashed           | Same rows as with the identifier but it is replaced by a keyed hash (HMAC-SHA256).          |

General contents
- Setup
- Bronze layer: Raw data Ingestion
- Silver layer: Combination, Validation, Calculation of remaining lease and resale price anomalies
- Gold layer: Resale identifier and subsequent de-identification
- ETL Lifecycle profiling
- Deliverables Inventory

How to - Run
1. In your repository terminal, run `pip install -r requirements.txt` (Implementation was done on Python 3.13, but all functions should work on 3.11 and later)
2. In the repository terminal, run `copy .env.example .env`, then place a real secret in `HASH_PEPPER` (this can be achieved by using the following command in the terminal `python -c "import secrets; print(secrets.token_hex(32))"`, and then pasting the result in `HASH_PEPPER` where the `CHANGE_ME` is.
NOTE: Hashes generated will be different from the hashes in the commited files which is the expected behavior due to the method chosen to adhere to requirements. Production behavior and decision making will be discussed in the relevant sections.
3. Ensure that the intended configurations are set in `src/config.yaml`.
NOTE: All paths, URLs, thresholds and rule configurations come from `src/config.yaml`.
4. Run all cells from top to bottom. Ingestion will skip files that already exists to ensure that re-runs are safe.

## 1. Setup
### Import required libraries, load config, config helpers and relevant variables

In [1]:
import logging
import sys
from pathlib import Path

import polars as pl
from IPython.display import display

# Find the repository root and make src importable
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"config"/"config.yaml").exists())
sys.path.insert(0, str(REPO_ROOT/"src"))

from src.config import load_config, get_hash_secret
from src.logging_setup import setup_logging
from src.bronze.ingest import run_ingestion_pipeline
from src.silver.combine import run_combine, write_combined_df, summarize_ingestion_sizes
from src.silver.profiling import generate_ydata_profile_reference_dataset, profile_stage, generate_ydata_profile
from src.silver.validation import (
    build_authoritative_dataset_value_set, run_validation, quarantine_reason_counts, summarize_rejections, write_validation_outputs
)
from src.silver.calc_remaining_lease import calculate_remaining_lease
from src.silver.anomaly_resale_prices import flag_resale_price_anomalies, summarize_anomalies_report, split_anomalies
from src.gold.identifier import (
    add_resale_identifier, summarize_identifier_collisions, write_transformed_identifier_df
)
from src.gold.deidentify import deidentify_identifier, write_hashed_output_df

# Ensures that polars shows every column in outputs
pl.Config.set_tbl_cols(-1)

# Sets how many rows polars will show in outputs
pl.Config.set_tbl_rows(30)

# Reads config and .env
cfg = load_config()

setup_logging(cfg)

# Check whether hash secret was set correctly
get_hash_secret(cfg=cfg)
print("HASH_PEPPER set and long enough.")

HASH_PEPPER set and long enough.


In [2]:
# Key configurations for the run (all from config/config.yaml) - Please ensure they are set as expected
display(pl.DataFrame({
    "Settings": [
        "collection_id",
        "scope (inclusive)",
        "authoritative month",
        "validated_categorical_columns",
        "lease term/ as of data"
    ],
    "Value": [
        cfg.require("source.collection_id"),
        "{start} to {end}".format(start=cfg.require("scope.start_year_month"), end=cfg.require("scope.end_year_month")),
        "{month}".format(month=cfg.require("reference.authoritative_dataset_month")),
        ", ".join(cfg.require("validation.categorical_columns")),
        "{full_term} years / {as_of_date}".format(
            full_term=cfg.require("calc_remaining_lease.lease_full_term_years"),
            as_of_date=cfg.require("calc_remaining_lease.lease_calc_period_end_date"))
    ]
}, schema={"Settings":pl.String, "Value": pl.String}, strict=False))

Settings,Value
str,str
"""collection_id""","""189"""
"""scope (inclusive)""","""2012-01 to 2017-12"""
"""authoritative month""","""2012-01"""
"""validated_categorical_columns""","""town, flat_type, flat_model, s…"
"""lease term/ as of data""","""99 years / 2027-01-01"""


## 2. Bronze Layer
### 2.1 Raw data Ingestion
Ingest raw data and output bronze file group.
The collection's child datasets are discovered from the data.gov.sg collection metadata. Then, each file is downloaded to `data_storage/bronze/raw` as is with no modifications at all.

* Downloads retry with backoff on network errors (429 and 5xx responses). Rate limit is honored with a delay configured in config.yaml.
* Files are written to a temporary `.part` file and renamed when complete, to avoid failed downloads leaving a truncated file.
* Existing files are skipped, to ensure runs are idempotent.

In [3]:
result = run_ingestion_pipeline(cfg=cfg)

# Shows which datasets were downloaded/skipped/failed
print({k: len(v) for k, v in result.items()})

assert not result["failed"],  "Datasets failed to download: {result}".format(result=result['failed'])

2026-10-08 14:19:10,159 - INFO - src.bronze.ingest: Found 5 dataset(s) to ingest
2026-10-08 14:19:11,695 - INFO - src.bronze.ingest: Saved d_8b84c4ee58e3cfc0ece0d773c8ca6abc -> C:\Users\bishr\202610_zenika_assignment\data_storage\bronze\raw\d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv
2026-10-08 14:19:22,883 - INFO - src.bronze.ingest: Saved d_43f493c6c50d54243cc1eab0df142d6a -> C:\Users\bishr\202610_zenika_assignment\data_storage\bronze\raw\d_43f493c6c50d54243cc1eab0df142d6a.csv
2026-10-08 14:19:33,496 - INFO - src.bronze.ingest: Saved d_2d5ff9ea31397b66239f245f57751537 -> C:\Users\bishr\202610_zenika_assignment\data_storage\bronze\raw\d_2d5ff9ea31397b66239f245f57751537.csv
2026-10-08 14:19:44,717 - INFO - src.bronze.ingest: Saved d_ebc5ab87086db484f88045b47411ebc5 -> C:\Users\bishr\202610_zenika_assignment\data_storage\bronze\raw\d_ebc5ab87086db484f88045b47411ebc5.csv
2026-10-08 14:19:55,319 - INFO - src.bronze.ingest: Saved d_ea9ed51da2787afaf8e51f827c304208 -> C:\Users\bishr\202610_zenik

{'downloaded': 5, 'skipped': 0, 'failed': 0}


In [4]:
# Raw output files and their sizes (in mb)
raw_dir = cfg.path("bronze_raw")
display(pl.DataFrame([{
    "file": p.name,
    "size_mb": round(p.stat().st_size / 1e6, 2)} for p in sorted(raw_dir.glob("*.csv"))
]))

file,size_mb
str,f64
"""d_2d5ff9ea31397b66239f245f5775…",4.16
"""d_43f493c6c50d54243cc1eab0df14…",29.37
"""d_8b84c4ee58e3cfc0ece0d773c8ca…",23.96
"""d_ea9ed51da2787afaf8e51f827c30…",3.07
"""d_ebc5ab87086db484f88045b47411…",22.64


## 3. Silver Layer
### 3.1 Combination into main dataset

- Every file is read as **strings** (no type inference), so values remain exactly as received (no modifications requirement).
- The file containing the authoritative month (**configured in config.yaml but 2012-01 as per requirements**) defines the authoritative schema (columns and order). Columns that exist only in the later files (such as `remaining_lease` from analysis), are left out of the main dataset and remain intact unchanged in Bronze.
- A `source_file` column saves where each row has come from.
- Nothing is filtered or de-duplicated yet here.

In [5]:
# Write set to False to show combined dataframe write path
combined_df, frames = run_combine(
    cfg=cfg,
    write=False
)
combined_path = write_combined_df(
    cfg=cfg,
    combined_df=combined_df,
)

# Showcases per-file and combined sizes: evidence for engine choice below
display(summarize_ingestion_sizes(
    cfg=cfg,
    frames=frames,
    combined_df=combined_df,
    parquet_path=combined_path,
))

2026-10-08 14:20:05,460 - INFO - src.silver.combine: Read d_2d5ff9ea31397b66239f245f57751537.csv: 52203 rows, 10 cols
2026-10-08 14:20:05,479 - INFO - src.silver.combine: Read d_43f493c6c50d54243cc1eab0df142d6a.csv: 369651 rows, 10 cols
2026-10-08 14:20:05,495 - INFO - src.silver.combine: Read d_8b84c4ee58e3cfc0ece0d773c8ca6abc.csv: 242259 rows, 11 cols
2026-10-08 14:20:05,501 - INFO - src.silver.combine: Read d_ea9ed51da2787afaf8e51f827c304208.csv: 37153 rows, 11 cols
2026-10-08 14:20:05,518 - INFO - src.silver.combine: Read d_ebc5ab87086db484f88045b47411ebc5.csv: 287196 rows, 10 cols
2026-10-08 14:20:05,521 - INFO - src.silver.combine: Reference filename: d_43f493c6c50d54243cc1eab0df142d6a.csv: authoritative columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price']
2026-10-08 14:20:05,524 - WARNING - src.silver.combine: Dropping non-authoritative columns from reference (still kept in Bronze r

dataset,rows,cols,csv_size_mb,in_scope_rows
str,i64,i64,f64,i64
"""d_2d5ff9ea31397b66239f245f5775…",52203,10,4.16,52203
"""d_43f493c6c50d54243cc1eab0df14…",369651,10,29.37,3188
"""d_8b84c4ee58e3cfc0ece0d773c8ca…",242259,11,23.96,20509
"""d_ea9ed51da2787afaf8e51f827c30…",37153,11,3.07,37153
"""d_ebc5ab87086db484f88045b47411…",287196,10,22.64,0
"""COMBINED (total)""",988462,10,83.2,113053


In [6]:
# Check: No rows are added or lost during combination
assert combined_df.height == sum(df.height for df in frames.values()), "Row count drift has occurred during combination"

display(pl.DataFrame({
    "column": combined_df.columns,
}))

display("Combined data row count: " + str(combined_df.height))

display(combined_df.head(5))

column
str
"""month"""
"""town"""
"""flat_type"""
"""block"""
"""street_name"""
"""storey_range"""
"""floor_area_sqm"""
"""flat_model"""
"""lease_commence_date"""


'Combined data row count: 988462'

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file
str,str,str,str,str,str,str,str,str,str,str
"""2012-03""","""ANG MO KIO""","""2 ROOM""","""172""","""ANG MO KIO AVE 4""","""06 TO 10""","""45""","""Improved""","""1986""","""250000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-03""","""ANG MO KIO""","""2 ROOM""","""510""","""ANG MO KIO AVE 8""","""01 TO 05""","""44""","""Improved""","""1980""","""265000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""610""","""ANG MO KIO AVE 4""","""06 TO 10""","""68""","""New Generation""","""1980""","""315000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""474""","""ANG MO KIO AVE 10""","""01 TO 05""","""67""","""New Generation""","""1984""","""320000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""604""","""ANG MO KIO AVE 5""","""06 TO 10""","""67""","""New Generation""","""1980""","""321000""","""d_2d5ff9ea31397b66239f245f5775…"


#### **Engine choice for data operations**
The decision was originally between pySpark or Polars. Using pySpark already would simplify integrating the code into the targeted cloud infrastructure (AWS Glue), however, the size table above and the row count show that the full collection is small enough to process in memory on device. Spark would add a JVM dependency and start-up cost without any realistic benefit at the moment at this size.
Spark + Glue is the scale path and will be described in Part 2 Architecture.


### 3.2 Profiling the authoritative dataset (As per requirements: 2012-01)
The January 2012 rows define what "valid" values are for categorical columns. Therefore, they are profiled first with [ydata-profiling](https://github.com/ydataai/ydata-profiling), an open-source data profiling framework as required.
As ydata doesn't natively support Polars, the small authoritative slice is converted to pandas (where the numeric columns are cast within that profiling copy only so that the pipeline remains untouched at this point).
The HTML and JSON reports are written to `reports/profiling`, and the findings are written in `reports/data_profiling_report.md`.

In [7]:
# While this step shouldn't take long, if performance is bottle-necked, minimal can be set to True in generate_ydata_profile
# Takes combined_df and filters down to reference set parameters - 2012-01 for month
reference_report_paths = generate_ydata_profile_reference_dataset(
    cfg=cfg,
    combined_df=combined_df,
)
print(reference_report_paths)

2026-10-08 14:20:07,223 - INFO - visions.backends: Pandas backend loaded 2.3.3
2026-10-08 14:20:07,235 - INFO - visions.backends: Numpy backend loaded 2.3.5
2026-10-08 14:20:07,236 - INFO - visions.backends: Pyspark backend NOT loaded
2026-10-08 14:20:07,236 - INFO - visions.backends: Python backend loaded
2026-10-08 14:20:08,738 - INFO - src.silver.profiling: Profiling 1559 rows for 2012-01
100%|██████████| 10/10 [00:00<00:00, 101.12it/s]
2026-10-08 14:20:10,277 - INFO - matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-08 14:20:10,278 - INFO - matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-08 14:20:10,597 - INFO - matplotlib.category: Using categori

{'html': WindowsPath('C:/Users/bishr/202610_zenika_assignment/reports/profiling/profile_reference_2012-01.html'), 'json': WindowsPath('C:/Users/bishr/202610_zenika_assignment/reports/profiling/profile_reference_2012-01.json')}


Added a profiling step for the entire data to gather more insights before further steps.

In [8]:
# While this step shouldn't take long, if performance is bottle-necked, minimal can be set to True in generate_ydata_profile
silver_combined_stage_name = cfg.require("outputs.silver_combined_filename")
report_paths = generate_ydata_profile(
    cfg=cfg,
    df=combined_df,
    profile_name=silver_combined_stage_name
)
print(report_paths)

2026-10-08 14:20:14,287 - INFO - src.silver.profiling: Profiling 988462 rows for resale_combined
100%|██████████| 11/11 [00:08<00:00,  1.24it/s]
2026-10-08 14:20:34,700 - INFO - matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-08 14:20:34,740 - INFO - matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-08 14:20:36,496 - INFO - matplotlib.category: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-10-08 14:20:36,536 - INFO - matplotlib.category: Using categorical units to plot a list of strings that are all parsab

{'html': WindowsPath('C:/Users/bishr/202610_zenika_assignment/reports/profiling/profile_reference_resale_combined.html'), 'json': WindowsPath('C:/Users/bishr/202610_zenika_assignment/reports/profiling/profile_reference_resale_combined.json')}


In [9]:
# Lightweight, polars-native profiling (rows, nulls, distinct values per column), reused at every stage during design and implementation
stage_profiles = [profile_stage(df=combined_df, stage="silver_combined")]
display(stage_profiles[0])

stage,column,row_count,null_count,null_percentage,distinct_count
str,str,i64,i64,f64,i64
"""silver_combined""","""month""",988462,0,0.0,442
"""silver_combined""","""town""",988462,0,0.0,27
"""silver_combined""","""flat_type""",988462,0,0.0,8
"""silver_combined""","""block""",988462,0,0.0,2790
"""silver_combined""","""street_name""",988462,0,0.0,597
"""silver_combined""","""storey_range""",988462,0,0.0,25
"""silver_combined""","""floor_area_sqm""",988462,0,0.0,392
"""silver_combined""","""flat_model""",988462,0,0.0,34
"""silver_combined""","""lease_commence_date""",988462,0,0.0,58


### 3.3 Validation and quarantine
Determined rules are evaluated together,so a row that breaks several is quarantined once with every reason listed in `reject_reason`.

| **Rule**                                                                                        | **Reject reason**   |
|-------------------------------------------------------------------------------------------------|---------------------|
| Required column is null or blank                                                                | `MISSING_REQUIRED`  |
| Price, area or lease year not numeric, below its minimum, or lease starts after the sale year   | `INVALID_NUMERIC`   |
| Repeat of an earlier row (all columns except `source_file`; the first occurence is kept         | `DUPLICATE`         |
| town, flat_type, flat_model or storey_range not found in the 2012-01 data (nulls fail here too) | `INVALID_<COLUMN>`  |
| Month outside 2012-01 to 2012-12 (Inclusive)                                                    | `OUT_OF_SCOPE_DATE` |

Since as per the requirements, matching against the authoritative set is **strict**, exact value with no case folding or whitespace trimming is implemented (can be configured in config.yaml). The reference sets are bult from the full combined data, before any row is removed.

In [10]:
# Allowed values per validated column based on 2012-01 rows only
authoritative_value_sets = build_authoritative_dataset_value_set(
    cfg=cfg,
    combined_df=combined_df,
)
for column, values in authoritative_value_sets.items():
    print("{cols}: {len_values} allowed values -> {values}\n".format(
        cols=column,
        len_values=len(values),
        values=", ".join(str(v) for v in values),
    ))

2026-10-08 14:23:20,055 - INFO - src.silver.validation: Reference town: 26 distinct values
2026-10-08 14:23:20,056 - INFO - src.silver.validation: Reference flat_type: 7 distinct values
2026-10-08 14:23:20,056 - INFO - src.silver.validation: Reference flat_model: 13 distinct values
2026-10-08 14:23:20,057 - INFO - src.silver.validation: Reference storey_range: 12 distinct values


town: 26 allowed values -> ANG MO KIO, BEDOK, BISHAN, BUKIT BATOK, BUKIT MERAH, BUKIT PANJANG, BUKIT TIMAH, CENTRAL AREA, CHOA CHU KANG, CLEMENTI, GEYLANG, HOUGANG, JURONG EAST, JURONG WEST, KALLANG/WHAMPOA, MARINE PARADE, PASIR RIS, PUNGGOL, QUEENSTOWN, SEMBAWANG, SENGKANG, SERANGOON, TAMPINES, TOA PAYOH, WOODLANDS, YISHUN

flat_type: 7 allowed values -> 1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, MULTI-GENERATION

flat_model: 13 allowed values -> Adjoined flat, Apartment, Improved, Maisonette, Model A, Model A-Maisonette, Model A2, Multi Generation, New Generation, Premium Apartment, Simplified, Standard, Terrace

storey_range: 12 allowed values -> 01 TO 03, 04 TO 06, 07 TO 09, 10 TO 12, 13 TO 15, 16 TO 18, 19 TO 21, 22 TO 24, 25 TO 27, 28 TO 30, 31 TO 33, 34 TO 36



In [11]:
# Apply data validation rules
validated_df, quarantined_df = run_validation(
    cfg=cfg,
    combined_df=combined_df
)
print("Validated: {validated_row_count} rows | Quarantined: {quarantined_row_count} rows".format(
    validated_row_count=validated_df.height,
    quarantined_row_count=quarantined_df.height,
))

display(validated_df.head(10))

2026-10-08 14:23:20,106 - INFO - src.silver.validation: Reference town: 26 distinct values
2026-10-08 14:23:20,108 - INFO - src.silver.validation: Reference flat_type: 7 distinct values
2026-10-08 14:23:20,109 - INFO - src.silver.validation: Reference flat_model: 13 distinct values
2026-10-08 14:23:20,109 - INFO - src.silver.validation: Reference storey_range: 12 distinct values
2026-10-08 14:23:20,924 - INFO - src.silver.validation: General Data Quality rules validation: 104783 passed, 883679 quarantined
2026-10-08 14:23:20,925 - INFO - src.silver.validation: Validation: 104783 valid, 883679 quarantined


Validated: 104783 rows | Quarantined: 883679 rows


month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file
str,str,str,str,str,str,str,str,str,str,str
"""2012-06""","""ANG MO KIO""","""2 ROOM""","""172""","""ANG MO KIO AVE 4""","""04 TO 06""","""45""","""Improved""","""1986""","""259000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""2 ROOM""","""406""","""ANG MO KIO AVE 10""","""01 TO 03""","""44""","""Improved""","""1979""","""265000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""2 ROOM""","""508""","""ANG MO KIO AVE 8""","""04 TO 06""","""44""","""Improved""","""1980""","""266000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""2 ROOM""","""323""","""ANG MO KIO AVE 3""","""07 TO 09""","""44""","""Improved""","""1977""","""270000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""2 ROOM""","""314""","""ANG MO KIO AVE 3""","""10 TO 12""","""44""","""Improved""","""1978""","""272500""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""3 ROOM""","""476""","""ANG MO KIO AVE 10""","""01 TO 03""","""67""","""New Generation""","""1979""","""319000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""3 ROOM""","""466""","""ANG MO KIO AVE 10""","""04 TO 06""","""67""","""New Generation""","""1984""","""320000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""3 ROOM""","""604""","""ANG MO KIO AVE 5""","""01 TO 03""","""67""","""New Generation""","""1980""","""320000""","""d_2d5ff9ea31397b66239f245f5775…"
"""2012-06""","""ANG MO KIO""","""3 ROOM""","""605""","""ANG MO KIO AVE 5""","""04 TO 06""","""67""","""New Generation""","""1980""","""323000""","""d_2d5ff9ea31397b66239f245f5775…"


In [12]:
display(quarantine_reason_counts(
    cfg=cfg,
    quarantined_df=quarantined_df,
))
display(summarize_rejections(
    cfg=cfg,
    quarantined_df=quarantined_df,
    authoritative_value_sets=authoritative_value_sets,
))
display(quarantined_df.head(10))

reject_reason_name,row_count
str,u32
"""OUT_OF_SCOPE_DATE""",875409
"""INVALID_FLAT_MODEL""",293037
"""INVALID_STOREY_RANGE""",7947
"""DUPLICATE""",2014
"""INVALID_FLAT_TYPE""",279
"""INVALID_TOWN""",64
"""INVALID_NUMERIC""",51


column,value,row_count
str,str,u32
"""town""","""LIM CHU KANG""",64
"""flat_type""","""MULTI GENERATION""",279
"""flat_model""","""NEW GENERATION""",78898
"""flat_model""","""IMPROVED""",73589
"""flat_model""","""MODEL A""",70381
"""flat_model""","""SIMPLIFIED""",23258
"""flat_model""","""STANDARD""",17375
"""flat_model""","""MAISONETTE""",12215
"""flat_model""","""APARTMENT""",9901


month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,reject_reason
str,str,str,str,str,str,str,str,str,str,str,str
"""2012-03""","""ANG MO KIO""","""2 ROOM""","""172""","""ANG MO KIO AVE 4""","""06 TO 10""","""45""","""Improved""","""1986""","""250000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""2 ROOM""","""510""","""ANG MO KIO AVE 8""","""01 TO 05""","""44""","""Improved""","""1980""","""265000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""610""","""ANG MO KIO AVE 4""","""06 TO 10""","""68""","""New Generation""","""1980""","""315000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""474""","""ANG MO KIO AVE 10""","""01 TO 05""","""67""","""New Generation""","""1984""","""320000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""604""","""ANG MO KIO AVE 5""","""06 TO 10""","""67""","""New Generation""","""1980""","""321000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""154""","""ANG MO KIO AVE 5""","""01 TO 05""","""68""","""New Generation""","""1981""","""321000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""110""","""ANG MO KIO AVE 4""","""01 TO 05""","""67""","""New Generation""","""1978""","""323000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""445""","""ANG MO KIO AVE 10""","""01 TO 05""","""67""","""New Generation""","""1979""","""325000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""
"""2012-03""","""ANG MO KIO""","""3 ROOM""","""476""","""ANG MO KIO AVE 10""","""11 TO 15""","""67""","""New Generation""","""1979""","""328000""","""d_2d5ff9ea31397b66239f245f5775…","""INVALID_STOREY_RANGE"""


### 3.4 Calculate remaining lease as of 2027-01-01
The data only gives the lease start **year**, so the lease is assumed to start in January of that year and run 99 years. Remaining lease is measured from 2027-01-01, rounded down to whole months and split into years and months.
Since the remaining lease is measured from 2027-01-01 (January) and the start month for each lease commence date is also january, the months part is always 0 here (both of which can be configured in config.yaml).

Additionally, expired leases are set to 0 instead of showing negative values.

In [13]:
remaining_lease_df = calculate_remaining_lease(
    cfg=cfg,
    df=validated_df,
)

data_lease_commence_date_col = cfg.require("scope.data_lease_commence_date_col")
remaining_years_col = cfg.require("calc_remaining_lease.remaining_lease_years_col")
remaining_months_col = cfg.require("calc_remaining_lease.remaining_lease_months_col")
remaining_text_col = cfg.require("calc_remaining_lease.remaining_lease_text_col")

display(
    remaining_lease_df
    .group_by(
        data_lease_commence_date_col,
        remaining_years_col,
        remaining_months_col,
        remaining_text_col,
    ).agg(
        pl.len()
    )
    .rename({"len":"entry_count"})
    .unique()
    .sort(data_lease_commence_date_col).head(10)
)

lease_commence_date,remaining_lease_years,remaining_lease_months,remaining_lease_text,entry_count
str,i32,i32,str,u32
"""1966""",38,0,"""38 years 0 months""",9
"""1967""",39,0,"""39 years 0 months""",688
"""1968""",40,0,"""40 years 0 months""",196
"""1969""",41,0,"""41 years 0 months""",697
"""1970""",42,0,"""42 years 0 months""",1070
"""1971""",43,0,"""43 years 0 months""",681
"""1972""",44,0,"""44 years 0 months""",597
"""1973""",45,0,"""45 years 0 months""",756
"""1974""",46,0,"""46 years 0 months""",1374


### 3.5 Detection of potentially anomalous resale prices
Rows deemed to be anomalous are **flagged** but **not removed**, because an unusual resale price could still be a real sale and the requirements didn't indicate the necessity to quarantine these anomalous values.
The following two heuristics are used:
- **Peer-group outliers**: price is compared with other flats of the same town, flat type and year using the IQR rule (configurable). Groups that are too small fall back to flat type and year so thresholds stay stable.
- **Resale price per sqm outside set bounds**: bounds are configured in config.yaml. Present to catch errors that peer statistics may miss, for example if a whole grouping is wrong (albeit this is unlikely).

In [14]:
anomaly_df = flag_resale_price_anomalies(
    cfg=cfg,
    df=remaining_lease_df,
)
flag_col = cfg.require("anomaly.flag_col")

display(summarize_anomalies_report(
    cfg=cfg,
    df=anomaly_df
))
display(
    anomaly_df.filter(pl.col(flag_col))
    .sort("resale_price_num", descending=True)
    .head(10)
)

2026-10-08 14:23:21,433 - INFO - src.silver.anomaly_resale_prices: Flagged 3654 of 104783 rows as resale price anomalies


anomaly_reasons,row_count
str,u32
"""RESALE_PRICE_HIGH_FOR_PEERS""",2213
"""PRICE_PER_SQM_OUT_OF_BOUNDS""",1144
"""RESALE_PRICE_HIGH_FOR_PEERS;PR…",146
"""RESALE_PRICE_LOW_FOR_PEERS""",140
"""RESALE_PRICE_LOW_FOR_PEERS;PRI…",11


month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,source_file,remaining_lease_years,remaining_lease_months,remaining_lease_text,resale_price_num,year,resale_price_per_sqm,anomaly_reasons,is_anomaly
str,str,str,str,str,str,str,str,str,str,str,i32,i32,str,f64,str,f64,str,bool
"""2017-07""","""QUEENSTOWN""","""EXECUTIVE""","""148""","""MEI LING ST""","""19 TO 21""","""156""","""Apartment""","""1995""","""1160000""","""d_8b84c4ee58e3cfc0ece0d773c8ca…",67,0,"""67 years 0 months""",1.16e6,"""2017""",7435.897436,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2016-12""","""KALLANG/WHAMPOA""","""3 ROOM""","""57""","""JLN MA'MOR""","""01 TO 03""","""259""","""Terrace""","""1972""","""1150000""","""d_ea9ed51da2787afaf8e51f827c30…",44,0,"""44 years 0 months""",1.15e6,"""2016""",4440.15444,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2014-10""","""BISHAN""","""EXECUTIVE""","""194""","""BISHAN ST 13""","""22 TO 24""","""150""","""Maisonette""","""1987""","""1088888""","""d_2d5ff9ea31397b66239f245f5775…",59,0,"""59 years 0 months""",1.088888e6,"""2014""",7259.253333,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2017-03""","""BISHAN""","""EXECUTIVE""","""186""","""BISHAN ST 13""","""10 TO 12""","""146""","""Maisonette""","""1987""","""1088000""","""d_8b84c4ee58e3cfc0ece0d773c8ca…",59,0,"""59 years 0 months""",1.088e6,"""2017""",7452.054795,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2017-07""","""BISHAN""","""EXECUTIVE""","""190""","""BISHAN ST 13""","""19 TO 21""","""163""","""Maisonette""","""1987""","""1080000""","""d_8b84c4ee58e3cfc0ece0d773c8ca…",59,0,"""59 years 0 months""",1.08e6,"""2017""",6625.766871,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2015-03""","""KALLANG/WHAMPOA""","""3 ROOM""","""53""","""JLN MA'MOR""","""01 TO 03""","""280""","""Terrace""","""1972""","""1060000""","""d_ea9ed51da2787afaf8e51f827c30…",44,0,"""44 years 0 months""",1.06e6,"""2015""",3785.714286,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2017-12""","""KALLANG/WHAMPOA""","""3 ROOM""","""65""","""JLN MA'MOR""","""01 TO 03""","""249""","""Terrace""","""1972""","""1053888""","""d_8b84c4ee58e3cfc0ece0d773c8ca…",44,0,"""44 years 0 months""",1.053888e6,"""2017""",4232.481928,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2013-11""","""BISHAN""","""EXECUTIVE""","""190""","""BISHAN ST 13""","""19 TO 21""","""150""","""Maisonette""","""1987""","""1050000""","""d_2d5ff9ea31397b66239f245f5775…",59,0,"""59 years 0 months""",1.05e6,"""2013""",7000.0,"""RESALE_PRICE_HIGH_FOR_PEERS""",true
"""2015-04""","""BISHAN""","""EXECUTIVE""","""192""","""BISHAN ST 13""","""22 TO 24""","""149""","""Maisonette""","""1987""","""1050000""","""d_ea9ed51da2787afaf8e51f827c30…",59,0,"""59 years 0 months""",1.05e6,"""2015""",7046.979866,"""RESALE_PRICE_HIGH_FOR_PEERS""",true


In [15]:
# Applies anomaly configured action: 'flag' keep all rows, 'quarantine' remove flagged rows out
cleaned_df, anomaly_quarantined_df = split_anomalies(
    cfg=cfg,
    df=anomaly_df
)

# Check whether anomalous rows to be quarantined are available
if anomaly_quarantined_df.height:
    quarantined_df = pl.concat([quarantined_df, anomaly_quarantined_df], how="diagonal_relaxed")

print("Cleaned: {cleaned_row_count} rows | Quarantined: {quarantined_row_count}".format(
    cleaned_row_count=cleaned_df.height,
    quarantined_row_count=quarantined_df.height
))

Cleaned: 104783 rows | Quarantined: 883679


In [16]:
# While this step shouldn't take long, if performance is bottle-necked, minimal can be set to True in profile_reference_dataset
report_paths = generate_ydata_profile(
    cfg=cfg,
    df=anomaly_df,
    profile_name="silver_post_anomaly_detection"
)
print(report_paths)

2026-10-08 14:23:21,653 - INFO - src.silver.profiling: Profiling 104783 rows for silver_post_anomaly_detection
100%|██████████| 19/19 [00:00<00:00, 22.38it/s]
C:\Users\bishr\202610_zenika_assignment\venv\python\Lib\site-packages\ydata_profiling\model\pandas\discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[4 2 2 ... 5 4 4]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  discretized_df.loc[:, column] = self._discretize_column(
C:\Users\bishr\202610_zenika_assignment\venv\python\Lib\site-packages\ydata_profiling\model\pandas\discretize_pandas.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[4 4 4 ... 4 4 4]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  discretized_df.loc[:, column] = self._discretize_column(
2026-10-08 14:23:27,243 - I

{'html': WindowsPath('C:/Users/bishr/202610_zenika_assignment/reports/profiling/profile_reference_silver_post_anomaly_detection.html'), 'json': WindowsPath('C:/Users/bishr/202610_zenika_assignment/reports/profiling/profile_reference_silver_post_anomaly_detection.json')}


### 3.6 Write the cleaned and quarantined data outputs
Outputs will be written once after every silver layer step has concluded, so cleaned data includes the calculated remaining lease and anomaly columns whilst quarantined includes every rejected rows.
Additionally, the allowed values from the authoritative set are also saved as JSON for quick traceability.

In [17]:
# Every combined row must be either cleaned or quarantined (i.e. nothing is dropped)
assert cleaned_df.height + quarantined_df.height == combined_df.height, "Rows at some point during Silver layer operations, check after combine step."

silver_paths = write_validation_outputs(
    cfg=cfg,
    valid_df=cleaned_df,
    quarantined_df=quarantined_df,
    authoritative_value_sets=authoritative_value_sets
)

print(silver_paths)

2026-10-08 14:23:53,304 - INFO - src.silver.validation: Wrote valid -> C:\Users\bishr\202610_zenika_assignment\data_storage\silver\cleaned\resale_cleaned.parquet
2026-10-08 14:23:53,305 - INFO - src.silver.validation: Wrote quarantined -> C:\Users\bishr\202610_zenika_assignment\data_storage\silver\quarantined\resale_quarantined.parquet
2026-10-08 14:23:53,306 - INFO - src.silver.validation: Wrote reference -> C:\Users\bishr\202610_zenika_assignment\reports\profiling\authoritative_reference_values.json


{'valid': WindowsPath('C:/Users/bishr/202610_zenika_assignment/data_storage/silver/cleaned/resale_cleaned.parquet'), 'quarantined': WindowsPath('C:/Users/bishr/202610_zenika_assignment/data_storage/silver/quarantined/resale_quarantined.parquet'), 'reference': WindowsPath('C:/Users/bishr/202610_zenika_assignment/reports/profiling/authoritative_reference_values.json')}


## 4. Gold Layer
### 4.1 Resale Identifier
Built from the validated data with the following structure: `S + BBB + PPP + YYMM + T`

| **Part** | **Rule**                                                                              | **Example**            |
|----------|---------------------------------------------------------------------------------------|------------------------|
| S        | Constant prefix                                                                       | S                      |
| BBB      | First 3 digits of `block` after removing non-digits, left-padded with 0               | Block `19` gives `019` |
| PPP      | First 3 digits of the average resale price of the (year-month, town, flat_type) group | `S$230,000` gives `230`    |
| YYMM     | year-month of the row                                                                 | `2012-01` gives `1201` |
| T        | First letter of the town                                                              | Ang Mo Kio gives `A`   |

Multiple rows may share the same identifier, therefore, uniqueness is enforced **without dropping any record as per requirements**. Rows that collide get a sequence suffix (-001, 002,...), ranked by a fixed tie-break order and then by every other column.
The result is deterministic and does not depend on the input row order.

In [18]:
identifier_df = add_resale_identifier(
    cfg=cfg,
    validated_df=cleaned_df
)

# Summary of how many rows collide - i.e. need a suffix
display(pl.DataFrame([summarize_identifier_collisions(
    cfg=cfg,
    df=identifier_df
)]))

id_col = cfg.require("identifier.resale_id_col")
data_month_col = cfg.require("scope.data_month_col")
data_block_col = cfg.require("scope.data_block_col")
data_town_col = cfg.require("scope.data_town_col")
data_resale_price_col = cfg.require("scope.data_resale_price_col")

display(
    identifier_df.select(
        data_block_col,
        data_resale_price_col,
        data_month_col,
        data_town_col,
        id_col
    ).head(10)
)

2026-10-08 14:23:53,472 - INFO - src.gold.identifier: Added resale_identifier to 104783 rows


rows,suffixed_identifier_count,colliding_identifiers_count,largest_group_size
i64,i64,i64,i64
104783,23639,10687,14


block,resale_price,month,town,resale_identifier
str,str,str,str,str
"""172""","""259000""","""2012-06""","""ANG MO KIO""","""S1722661206A"""
"""406""","""265000""","""2012-06""","""ANG MO KIO""","""S4062661206A"""
"""508""","""266000""","""2012-06""","""ANG MO KIO""","""S5082661206A"""
"""323""","""270000""","""2012-06""","""ANG MO KIO""","""S3232661206A"""
"""314""","""272500""","""2012-06""","""ANG MO KIO""","""S3142661206A"""
"""476""","""319000""","""2012-06""","""ANG MO KIO""","""S4763521206A"""
"""466""","""320000""","""2012-06""","""ANG MO KIO""","""S4663521206A-001"""
"""604""","""320000""","""2012-06""","""ANG MO KIO""","""S6043521206A-001"""
"""605""","""323000""","""2012-06""","""ANG MO KIO""","""S6053521206A"""


In [19]:
# Additional checks to ensure that 1) nothing was dropped 2) every identifier is unique 3) every identifier matches the format
# Note that many of these checks are built directly within the functions as well
assert identifier_df.height == cleaned_df.height
assert identifier_df[id_col].n_unique() == identifier_df.height
# Check format with regex - Note that this will fail if relevant configurations are changed in config.yaml
assert identifier_df[id_col].str.contains(r"^S\d{10}[A-Z](-\d{3,})?$").all(), "Identifier format has been violated"

transformed_identifier_path = write_transformed_identifier_df(
    cfg=cfg,
    transformed_df=identifier_df
)
print(transformed_identifier_path)

2026-10-08 14:23:53,658 - INFO - src.gold.identifier: Wrote transformed output -> C:\Users\bishr\202610_zenika_assignment\data_storage\gold\transformed\resale_unique_identifiers.parquet


C:\Users\bishr\202610_zenika_assignment\data_storage\gold\transformed\resale_unique_identifiers.parquet


### 4.2 Deidentify the identifier from 4.1
The goal is to replace the identifier created in 3.1 with an **HMAC-SHA256** hash keyed with a secret (the pepper from `HASH_PEPPER`):
This method was determined as so to fulfill the requirements:
- **Be Irreversible**: HMAC-SHA256 is a one-way function
- **Preserve Uniqueness**: deterministic and collision-resistant, which means unique identifiers stay unique.
- **Protect against dictionary attacks**: since the identifier space is small and structured, a plain or publicly "salted" hash could be brute-forced by hashing every candidate (an attack form made even easier with AI Agent Swarms...). However, with this implementation, without the secret key, an attacker cannot do that.
    - Note: While yes, a random per-row salt would stop dictionary attacks too, it would also break the uniqueness requirement. Therefore, that wasn't used.

The secret is read from the environment and is never stored in the repository, the config or the notebook output.

In [20]:
hashed_df = deidentify_identifier(
    cfg=cfg,
    transformed_df=identifier_df
)

hash_col = cfg.require("deidentify.hash_id_col")
display(hashed_df.select(
        data_block_col,
        data_resale_price_col,
        data_month_col,
        data_town_col,
        hash_col
).head(10))

2026-10-08 14:23:53,973 - INFO - src.gold.deidentify: De-identified 104783 identifiers with HMAC-SHA256


block,resale_price,month,town,resale_identifier_hashed
str,str,str,str,str
"""172""","""259000""","""2012-06""","""ANG MO KIO""","""dc09e0cd8fe41343f189104d43dba1…"
"""406""","""265000""","""2012-06""","""ANG MO KIO""","""8ffe451401fd614cd0b9529c674653…"
"""508""","""266000""","""2012-06""","""ANG MO KIO""","""cd1dd3e119a6aabdf61d8bb084c14c…"
"""323""","""270000""","""2012-06""","""ANG MO KIO""","""b6b57c323a85ace1b95a6e831d2dfd…"
"""314""","""272500""","""2012-06""","""ANG MO KIO""","""f00feba6dc5908f71c25822f7f90d2…"
"""476""","""319000""","""2012-06""","""ANG MO KIO""","""f2065a7a5188c7f3fccc3d5533d07a…"
"""466""","""320000""","""2012-06""","""ANG MO KIO""","""5e240ee26878a47eecbc2219529a1e…"
"""604""","""320000""","""2012-06""","""ANG MO KIO""","""d280e5b738a485fc0a0e80dccc70a8…"
"""605""","""323000""","""2012-06""","""ANG MO KIO""","""ea5abab6017bf42a604028e9d78089…"


In [21]:
# Checks to ensure 1) same number of rows 2) every hash is unique 3) 64 hex characters 4) plain identifier is removed
assert hashed_df.height == identifier_df.height
assert hashed_df[hash_col].n_unique() == hashed_df.height
assert hashed_df[hash_col].str.contains(r"^[0-9a-f]{64}$").all()
assert id_col not in hashed_df.columns

hashed_path = write_hashed_output_df(
    cfg=cfg,
    hashed_df=hashed_df
)
print(hashed_path)

2026-10-08 14:23:54,159 - INFO - src.gold.deidentify: Wrote hashed output -> C:\Users\bishr\202610_zenika_assignment\data_storage\gold\hashed\resale_hashed_identifiers.parquet


C:\Users\bishr\202610_zenika_assignment\data_storage\gold\hashed\resale_hashed_identifiers.parquet


### 5. ETL Lifecycle profiling
To ensure evolving data insights have been analyzed throughout the ETL pipeline design and implementation lifecycle stages, the same profile was taken at every stage so the profiling report can describe how the data behaves from Bronze Layer to Gold Layer, including data in the quarantine bucket.
Additionally, row counts are reconciled to showcase that no data was lost.

In [22]:
stage_profile = pl.concat([
    stage_profiles[0],
    profile_stage(df=cleaned_df, stage="silver_cleaned"),
    profile_stage(df=quarantined_df, stage="silver_quarantined"),
    profile_stage(df=identifier_df, stage="gold_transformed_identified"),
    profile_stage(df=hashed_df, stage="gold_hashed"),
])

# Add to profiling report folder
stage_profile.write_csv(cfg.path("profiling", create=True) / "stage_profile.csv")

display(
    stage_profile.select(
        "stage",
        "row_count"
    ).unique()
)

# Example: Show nulls and distinct values per column at each stage - columns that weren't present at a stage will be null
display(
    stage_profile.pivot(
        on="stage",
        index="column",
        values="distinct_count",
    )
)

stage,row_count
str,i64
"""silver_cleaned""",104783
"""silver_combined""",988462
"""silver_quarantined""",883679
"""gold_transformed_identified""",104783
"""gold_hashed""",104783


column,silver_combined,silver_cleaned,silver_quarantined,gold_transformed_identified,gold_hashed
str,i64,i64,i64,i64,i64
"""month""",442,69,439,69,69
"""town""",27,26,27,26,26
"""flat_type""",8,7,8,7,7
"""block""",2790,2170,2790,2170,2170
"""street_name""",597,522,597,522,522
"""storey_range""",25,12,25,12,12
"""floor_area_sqm""",392,167,391,167,167
"""flat_model""",34,13,34,13,13
"""lease_commence_date""",58,50,58,50,50


### Deliverables Inventory
Short summary as to where to find and what makes up the five required output groups.

In [23]:
groups = {
    "Raw": "bronze_raw",
    "Cleaned": "silver_cleaned",
    "Quarantined": "silver_quarantined",
    "Transformed": "gold_transformed",
    "Hashed": "gold_hashed"
}

inventory = []

for group, path_key in groups.items():
    for p in sorted(cfg.path(path_key).glob("*")):
        if p.is_file():
            if p.suffix == ".parquet":
                row_count = pl.scan_parquet(p).select(pl.len()).collect().item()
            elif p.suffix == ".csv":
                row_count = pl.scan_csv(p).select(pl.len()).collect().item()
            else:
                continue
            inventory.append({
                "Output group": group,
                "file": str(p.relative_to(REPO_ROOT)),
                "size_mb": round(p.stat().st_size / 1e6, 2),
                "rows": row_count,
            })

display(pl.DataFrame(inventory))

Output group,file,size_mb,rows
str,str,f64,i64
"""Raw""","""data_storage\bronze\raw\d_2d5f…",4.16,52203
"""Raw""","""data_storage\bronze\raw\d_43f4…",29.37,369651
"""Raw""","""data_storage\bronze\raw\d_8b84…",23.96,242259
"""Raw""","""data_storage\bronze\raw\d_ea9e…",3.07,37153
"""Raw""","""data_storage\bronze\raw\d_ebc5…",22.64,287196
"""Cleaned""","""data_storage\silver\cleaned\re…",1.28,104783
"""Quarantined""","""data_storage\silver\quarantine…",4.88,883679
"""Transformed""","""data_storage\gold\transformed\…",1.66,104783
"""Hashed""","""data_storage\gold\hashed\resal…",4.98,104783
